#   Explorando a Arrecadação de Impostos no Brasil ($\text{2000-2026}$)

##  Dependências e Configuração
-   `numpy`, `pandas`: tratamento de dados numéricos, conversões, utilização de `pandas.DataFrame` enquanto estrutura de dados base para o trabalho;
-   `chardet`: identificação *on-line* de *charset encodings* para lidar com dados não tão limpos;
-   `altair`, `calendar`: visualizações; interface temporal

In [ ]:
import numpy    as np;
import pandas   as pd;
import altair   as alt;
import chardet
import calendar

#   dados
DATA: str = r"data/arrecadacao-estado.csv"


ValueError: 
To use the 'notebook' renderer, you must install the vega package
and the associated Jupyter extension.
See https://altair-viz.github.io/getting_started/installation.html
for more information.


## $\S.1 \,$ Leitura dos Dados

In [ ]:
#   detectar o character set utilizado nos dados
with open(DATA, "rb") as f:
    raw_data = f.read(10000);
    result = chardet.detect(raw_data);
    encoding = result['encoding'];
    print(f"charset encoding: {encoding}");

#   1.  Leitura e extração dos dados
#   ler com o encoding detectado
df = pd.read_csv(r"data/arrecadacao-estado.csv", encoding=encoding, sep=";");
print(df.head(3));

charset encoding: ISO-8859-1
    Ano      Mês  UF IMPOSTO SOBRE IMPORTAÇÃO IMPOSTO SOBRE EXPORTAÇÃO  \
0  2000  Janeiro  AC                      231                        0   
1  2000  Janeiro  AL                   475088                    33873   
2  2000  Janeiro  AM                 11679405                        0   

  IPI - FUMO IPI - BEBIDAS IPI - AUTOMÓVEIS IPI - VINCULADO À IMPORTACAO  \
0     292096             0                0                          167   
1    1329338        812470                0                       141735   
2    1507146       1791471            27796                      4414483   

  IPI - OUTROS  ... REFIS PAES RETENÇÃO NA FONTE - LEI 10.833, Art. 30  \
0         1558  ...   NaN  NaN                                     NaN   
1      3676847  ...   NaN  NaN                                     NaN   
2      1800346  ...   NaN  NaN                                     NaN   

  PAGAMENTO UNIFICADO OUTRAS RECEITAS ADMINISTRADAS DEMAIS RECEITAS  \
0

## $\S2. \,$ Pré-Processamento
### $\S2.1 \,$ Limpeza dos Dados
Transformação de *strings* de moeda ($\text{R\$}$) para valores numéricos

In [ ]:
def clean_currency(col):
    # Se a coluna já for numérica, retorna sem alteração
    if pd.api.types.is_numeric_dtype(col):
        return col

    # Converte para string e remove espaços
    col = col.astype(str).str.strip()
    
    # Remove qualquer caractere que não seja dígito, vírgula ou ponto
    col = col.str.replace(r'[^\d,.]', '', regex=True)
    
    # Substitui vírgula (separador decimal) por ponto
    col = col.str.replace(',', '.', regex=False)
    
    # Função para converter cada valor, tratando pontos de milhar
    def convert_value(x):
        if x == '' or pd.isna(x):
            return np.nan
        # Separa por pontos
        parts = x.split('.')
        if len(parts) == 1:
            # Não tem ponto decimal (ex: '47953915')
            return float(parts[0])
        else:
            # Último segmento é a parte decimal
            decimal = parts[-1]
            # Junta os segmentos anteriores (removendo pontos de milhar)
            integer_part = ''.join(parts[:-1])
            # Reconstrói o número com ponto decimal
            return float(integer_part + '.' + decimal)
    
    return col.apply(convert_value)

#   Converte as strings que representam R$ para float
cols_to_clean       = [c for c in df.columns if c not in ['Ano', 'Mês', 'UF']];
df[cols_to_clean]   = df[cols_to_clean].apply(clean_currency);

### $\S2.2. \,$ Criação de Novas Colunas

In [ ]:
#   criar associações mês -> número, número -> mês
meses_pt = ['Janeiro', 'Fevereiro', 'Março', 'Abril', 'Maio', 'Junho',
            'Julho', 'Agosto', 'Setembro', 'Outubro', 'Novembro', 'Dezembro']

month_to_number = {mes: str(i+1).zfill(2) for i, mes in enumerate(meses_pt)}

#   criação de novas colunas
df['Mês_Num'] = df['Mês'].map(month_to_number)  # cria coluna '01' a '12'
df['Data'] = pd.to_datetime(df['Ano'].astype(str) + '-' + df['Mês_Num'] + '-01')

### $\S2.3. \,$ Transformação dos Dados
Adequação dos dados ao formato `melt` necessário à biblioteca `altair`.

In [ ]:
#   colunas identificadoras
id_vars = ['Ano', 'Mês', 'UF', 'Mês_Num', 'Data']

#   colunas valor
value_vars = [c for c in df.columns if c not in id_vars]

#   -> melt
df_melt = df.melt(id_vars=id_vars, value_vars=value_vars, var_name='Imposto', value_name='Valor')
df_melt = df_melt.dropna(subset=['Valor']).query('Valor > 0')  # Remove zeros/NaNs

print(df_melt.shape)

(215309, 7)


## $\S 3. \,$ Visualizações
### $\S 3.1. \,$ Série Temporal: Evolução da Arrecação Total

In [40]:
import numpy as np
from scipy import stats  # ou use numpy.polyfit

# 1. Calcular a regressão linear no pandas
# Converter Data para número de dias desde a primeira data (para regressão)
trend['Data_num'] = (trend['Data'] - trend['Data'].min()).dt.days

# Ajustar uma reta: Valor = a * Data_num + b
slope, intercept, r_value, p_value, std_err = stats.linregress(trend['Data_num'], trend['Valor'])

# Criar um DataFrame com a reta ajustada
trend['Reg_linear'] = slope * trend['Data_num'] + intercept

# 2. Criar os gráficos (sem transform_regression)
base = alt.Chart(trend).encode(x=alt.X('Data:T', title='Data'))

# Série original
orig = base.mark_line(color='lightgray', opacity=0.3).encode(y='Valor:Q')

# Banda (mín/máx móvel)
band = base.mark_area(opacity=0.15, color='steelblue').encode(
    y='Min_12:Q',
    y2='Max_12:Q'
)

# Média móvel
ma = base.mark_line(color='steelblue', strokeWidth=2).encode(y='MA_12:Q')

# Reta de regressão (calculada no pandas)
reg = base.mark_line(color='orange', strokeWidth=2).encode(y='Reg_linear:Q')

# Combinação
chart = (band + ma + reg + orig).properties(
    title='Arrecadação – banda de 12 meses, média móvel e regressão linear',
    width=700,
    height=400
).interactive()

chart.show()  # ou chart.save('chart.html')

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 2.2.6
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

### $\S 3.2. \,$ Stacked Area: participação relativa dos tributos
O que aconteceu em 2004?

In [ ]:
# Selecionar apenas os maiores impostos para não poluir
top_impostos = ['IRPF', 'IRPJ - DEMAIS EMPRESAS', 'COFINS - DEMAIS', 
                'CONTRIBUIÇÃO PARA O PIS/PASEP - DEMAIS', 'CSLL - DEMAIS',
                'IPI - AUTOMÓVEIS', 'IMPOSTO SOBRE IMPORTAÇÃO']

df_plot = df_melt[df_melt['Imposto'].isin(top_impostos)]
# Agregar por Data e Imposto (soma nacional)
df_agg = df_plot.groupby(['Data', 'Imposto'])['Valor'].sum().reset_index()

# Gráfico de áreas (stacked normalizado)
area_chart = alt.Chart(df_agg).mark_area(opacity=0.7).encode(
    x=alt.X('Data:T', title='Data'),
    y=alt.Y('Valor:Q', title='Arrecadação (R$)', stack='normalize'),
    color=alt.Color('Imposto:N', legend=alt.Legend(columns=2)),
    tooltip=['Data', 'Imposto', 'Valor']
).properties(
    title='Participação relativa dos principais tributos',
    width=700,
    height=400
).interactive()

# Linha vertical para marcar o início de 2004 (01/01/2004)
linha_2004 = alt.Chart(pd.DataFrame({'Data': ['2004-01-01']})).mark_rule(
    color='red', 
    strokeWidth=2,
    strokeDash=[5, 3]  # Linha tracejada para não poluir
).encode(
    x='Data:T'
)

# Unir os dois gráficos
(area_chart + linha_2004)

alt.LayerChart(...)

### $\S 3.3. \,$ Mapa de Calor: Sazonalidade por Estado

In [ ]:
# Selecionar um imposto específico (ex: IRPF) ou total. Vou usar total por UF/Mês.
# Agregar por UF e Mês (média ou total ao longo dos anos)
heat_data = df_melt.groupby(['UF', 'Mês_Num'])['Valor'].mean().reset_index()
# Ordenar meses corretamente
heat_data['Mês'] = pd.to_datetime(heat_data['Mês_Num'] + '-01').dt.month

alt.Chart(heat_data).mark_rect().encode(
    x=alt.X('Mês:O', title='Mês (1=Jan)'),
    y=alt.Y('UF:N', title='Estado'),
    color=alt.Color('Valor:Q', scale=alt.Scale(scheme='redyellowblue'), title='Média Arrec.' ),
    tooltip=['UF', 'Mês', 'Valor']
).properties(
    title='Média mensal de arrecadação por Estado (todos os impostos)',
    width=500,
    height=400
).interactive()

/tmp/ipykernel_16675/2283440565.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  heat_data['Mês'] = pd.to_datetime(heat_data['Mês_Num'] + '-01').dt.month


OutOfBoundsDatetime: Out of bounds nanosecond timestamp: 01-01, at position 0